In [1]:
import torch
import torch.nn as nn

from fvcore.nn import FlopCountAnalysis
from contextlib import redirect_stderr
import io

def print_param_flops(net, input_shape):
    x = torch.randn(1, *input_shape).to("cuda")
    params = sum(p.numel() for p in net.parameters() if p.requires_grad)

    with redirect_stderr(io.StringIO()):
        flops = FlopCountAnalysis(net, (x,))
        flops_amount = flops.total()

    print(f"Parameters: {params/1e6:.2f} M,\tFLOPs: {flops_amount/1e9:.2f} G")

In [4]:
from net.modules.encoder import Encoder
from net.modules.decoder import Decoder
from net.modules.moga import MogaBlock

In [5]:
x_0 = torch.randn(1, 16, 256, 256)

In [6]:
encoder = Encoder(
    in_channels = x_0.shape[1],
    kernel_sizes = [3, 3, 3],
    features = [32, 64, 128],
    strides = [1, 2, 2],
    maxpools = [True, False, False],
    dropouts = [0.1, 0.1, 0.1],
    norm_name="batch",
    act_name=("leakyrelu", {"inplace": True, "negative_slope": 0.01}),
    block = MogaBlock, #nn.Identity,
    spatial_dims=2,
)

x, skips = encoder(x_0)

In [7]:
print("in shape:", x_0.shape)
print("out shape:", x.shape)
for i, skip in enumerate(skips):
    print(f"skip {i} shape:", skip.shape)

in shape: torch.Size([1, 16, 256, 256])
out shape: torch.Size([1, 128, 64, 64])
skip 0 shape: torch.Size([1, 16, 256, 256])
skip 1 shape: torch.Size([1, 32, 256, 256])
skip 2 shape: torch.Size([1, 64, 128, 128])


In [8]:
decoder = Decoder(
    in_channels = x.shape[1],
    features = [64, 32, x_0.shape[1]],
    block = MogaBlock, #nn.Identity,
    up_strides = [2, 2, 1],
    up_transpose=False,
    spatial_dims=2,
)

x = decoder(x, skips)
print("out shape:", x.shape)

out shape: torch.Size([1, 16, 256, 256])


In [10]:
# from net.models.baseline import Net as BaselineModel
from net import BaselineModel

In [11]:
x_0 = torch.randn(1, 1, 256, 256)

net = BaselineModel(
    in_channels=x_0.shape[1], 
    out_channels=9, 
    features=[64, 128, 256], 
    kernel_size=[3, 3, 3], 
    stride=[2, 2, 2],
    maxpools = [True, False, False],
    dropouts = [0.1, 0.1, 0.1],
    norm_name="batch",
    act_name=("leakyrelu", {"inplace": True, "negative_slope": 0.01}),
    block = MogaBlock, #nn.Identity,
    up_transpose=False,
).to("cuda")

print_param_flops(net, x_0.shape[1:])

Parameters: 1.82 M,	FLOPs: 11.51 G
